# WRDS Connection Starter

This notebook connects to WRDS using the `wrds` Python package and shows how to stream large query results in chunks.

Before running the connection cell, set your WRDS username in the environment as `WRDS_USERNAME`, or replace the placeholder in the cell. Do not hard-code your password in the notebook.

In [1]:
import importlib.util
import subprocess
import sys

# These packages should already be present in the project Conda environment.
required_packages = ["wrds", "pandas", "sqlalchemy", "pyarrow"]
missing_packages = [pkg for pkg in required_packages if importlib.util.find_spec(pkg) is None]

if missing_packages:
    print(f"Installing missing packages: {missing_packages}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])
else:
    print("All required packages are already installed.")

All required packages are already installed.


In [2]:
import os
from pathlib import Path

import pandas as pd
import wrds
from dotenv import load_dotenv
from sqlalchemy import text

load_dotenv()

WRDS_USERNAME = os.environ.get("WRDS_USERNAME")
WRDS_PASSWORD = os.environ.get("WRDS_PASSWORD")

if not WRDS_USERNAME:
    db = None
    print("Set WRDS_USERNAME before connecting, for example:")
    print("  export WRDS_USERNAME='your_wrds_username'")
    print("or add WRDS_USERNAME to a local .env file, then restart the kernel.")
elif WRDS_PASSWORD:
    db = wrds.Connection(wrds_username=WRDS_USERNAME, wrds_password=WRDS_PASSWORD)
    print("Connected to WRDS using credentials from the local environment.")
else:
    db = wrds.Connection(wrds_username=WRDS_USERNAME)
    print("Connected to WRDS after interactive password prompt.")

Loading library list...
Done
Connected to WRDS using credentials from the local environment.


## WRDS Smoke Checks

Run these cells only after `db` is connected. They keep the query sizes small so we can confirm access before pulling the full 2003-2014 CRSP dataset.

In [3]:
if db is None:
    print("Not connected. Set WRDS_USERNAME and rerun the connection cell first.")
else:
    libraries = db.list_libraries()
    print(f"Accessible libraries: {len(libraries)}")
    print("crsp available:", "crsp" in libraries)
    [library for library in libraries if "crsp" in library.lower()][:10]

Accessible libraries: 320
crsp available: True


In [4]:
if db is None:
    print("Not connected. Skipping CRSP table listing.")
elif "crsp" not in db.list_libraries():
    print("CRSP library is not available for this WRDS account.")
else:
    crsp_tables = db.list_tables(library="crsp")
    print(f"CRSP tables available: {len(crsp_tables)}")
    [table for table in crsp_tables if table in {"dsf", "msenames", "msedelist", "stocknames"}]

CRSP tables available: 433


## Tiny CRSP Daily Stock Query

This query pulls a very small CRSP sample for common shares on NYSE/AMEX/Nasdaq. If it works, the same columns and filters can be scaled into the full 2003-2014 market-data pull.

In [5]:
sample_query = """
select
    d.permno,
    d.date,
    n.ticker,
    n.comnam,
    n.shrcd,
    n.exchcd,
    d.prc,
    d.ret,
    d.vol,
    d.shrout
from crsp.dsf as d
join crsp.msenames as n
  on d.permno = n.permno
 and d.date between n.namedt and n.nameendt
where d.date between '2014-01-02' and '2014-01-10'
  and n.shrcd in (10, 11)
  and n.exchcd in (1, 2, 3)
limit 25
"""

if db is None:
    print("Not connected. Skipping sample CRSP query.")
elif "crsp" not in db.list_libraries():
    print("CRSP library is not available for this WRDS account.")
else:
    crsp_sample = db.raw_sql(sample_query, date_cols=["date"])
    print(f"Rows returned: {len(crsp_sample)}")
    display(crsp_sample.head())

Rows returned: 25


,permno,date,ticker,comnam,shrcd,exchcd,prc,ret,vol,shrout
0,10001,2014-01-02,EGAS,GAS NATURAL INC,11,2,8.04,0.001245,72900.0,10452.0
1,10025,2014-01-02,AEPI,A E P INDUSTRIES INC,11,3,53.04,0.003975,66074.0,5601.0
2,10026,2014-01-02,JJSF,J & J SNACK FOODS CORP,11,3,87.15,-0.016255,50930.0,18681.0
3,10028,2014-01-02,DGSE,D G S E COMPANIES INC,11,2,2.15,-0.035874,11300.0,12176.0
4,10032,2014-01-02,PLXS,PLEXUS CORP,11,3,42.78,-0.011781,145207.0,33787.0
